# Notebook 02: V-JEPA 2 Architecture Deep Dive

**Goal:** Understand the actual V-JEPA 2 code — ViT encoder, predictor, 3D RoPE, and how video is processed.

We'll read the real source code from `refs/vjepa2/` and build minimal runnable versions of each component.

---

## Architecture Overview

```
Video [B, C=3, T=16, H=256, W=256]
  │
  ├──► PatchEmbed3D (Conv3d)  →  [B, T/2 * H/16 * W/16, D]  =  [B, 2048, 1408]
  │        ↑ tubelet: 2×16×16
  │
  ├──► Mask ~90% of patches
  │
  ├──► Encoder (ViT-g): 40 transformer blocks with 3D RoPE
  │     → [B, ~205 visible tokens, 1408]  (only ~10% visible)
  │
  ├──► Predictor: 12 blocks, 384-dim
  │     → [B, ~1843 masked tokens, 1408]  (predicts 90%)
  │
  └──► Target Encoder (EMA): same ViT-g, processes ALL patches
        → [B, 2048, 1408]  (prediction targets)
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)

## 0.5 The 5W+H of V-JEPA 2's Architecture

### WHO designed V-JEPA 2?
**Adrien Bardes, Quentin Garrido, Jean Ponce, Yann LeCun** et al. at Meta FAIR (2025). It builds on I-JEPA (Assran et al., CVPR 2023) and V-JEPA (Bardes et al., 2024).

### WHAT is V-JEPA 2?
A **Vision Transformer (ViT)** pretrained with the JEPA objective on >1M hours of video. It converts video into rich, physics-aware latent representations. V-JEPA 2 uses ViT-giant (1.1B params), with 3D RoPE, SwiGLU FFN, and DeepNet weight rescaling.

### WHERE does V-JEPA 2 fit in the ML landscape?

| Category | Models | V-JEPA 2 Relationship |
|----------|--------|----------------------|
| **Foundation vision models** | DINOv2, SigLIP, CLIP | V-JEPA 2 is a video alternative — adds temporal understanding |
| **Video models** | VideoMAE, TimeSFormer | V-JEPA 2 predicts in latent space (not pixels) — better representations |
| **World models** | Dreamer, IRIS, Cosmos | V-JEPA 2-AC is a latent world model — 15x faster than pixel models |
| **VLAs** | RT-2, OpenVLA, pi0 | V-JEPA 2 provides features/world model for VLAs |

### WHEN was each component introduced?

| Year | Innovation | Paper |
|------|-----------|-------|
| 2020 | ViT (Vision Transformer) | Dosovitskiy et al. |
| 2021 | RoPE (Rotary Position Embeddings) | Su et al. |
| 2022 | SwiGLU for Transformers | Shazeer (Google) |
| 2022 | DeepNet (weight rescaling) | Wang et al. |
| 2023 | I-JEPA (images) | Assran et al. |
| 2024 | V-JEPA (video) | Bardes et al. |
| 2025 | V-JEPA 2 (3D RoPE + SwiGLU + scale) | Bardes et al. |

### WHY these specific architectural choices?

- **3D RoPE over learned positional embeddings:** RoPE encodes *relative* positions → generalizes to different video lengths and resolutions without retraining
- **SwiGLU over GELU MLP:** Gated activation allows selective information flow → better gradient dynamics for deep (40-layer) networks
- **DeepNet rescaling:** Without it, a 40-layer residual network accumulates variance → training diverges. Rescaling by $1/\sqrt{2L}$ stabilizes the forward pass
- **~90% masking:** Only 10% of tokens processed by encoder → 10x FLOPs savings during training; forces learning of rich, non-trivial representations

### HOW do all the pieces fit together?

$$\underbrace{\text{Video}}_{\mathbb{R}^{B \times 3 \times T \times H \times W}} \xrightarrow[\text{(Conv3d)}]{\text{PatchEmbed3D}} \underbrace{\text{Patch tokens}}_{\mathbb{R}^{B \times N \times D}} \xrightarrow{\text{mask}} \underbrace{\text{Visible}}_{\mathbb{R}^{B \times M \times D}}$$

Each ViT block applies:
$$\mathbf{x}' = \mathbf{x} + \underbrace{\text{Attention}_{\text{3D-RoPE}}}_{\text{communication}}(\text{LN}(\mathbf{x})), \quad \mathbf{x}'' = \mathbf{x}' + \underbrace{\text{SwiGLU}}_{\text{computation}}(\text{LN}(\mathbf{x}'))$$

The attention operation in matrix form:
$$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\!\left(\frac{\mathbf{R}_d\mathbf{Q} \cdot (\mathbf{R}_d\mathbf{K})^T}{\sqrt{d}}\right) \mathbf{V}$$

where $\mathbf{R}_d$ applies 3D rotary encoding per position axis (depth, height, width).

## 1. Video → Patches: PatchEmbed3D

A video `[B, 3, T, H, W]` is converted to patch tokens via a 3D convolution.

**Tubelet size: 2×16×16** means:
- 2 frames → 1 temporal token
- 16×16 pixels → 1 spatial token

For a 256×256 video with 16 frames:
- Temporal tokens: 16/2 = 8
- Spatial tokens per frame: (256/16) × (256/16) = 16 × 16 = 256
- Total tokens: 8 × 256 = **2048**

In [ ]:
class PatchEmbed3D(nn.Module):
    """Convert video to patch embeddings via 3D convolution.
    
    Source: refs/vjepa2/src/models/utils/patch_embed.py
    """
    def __init__(self, patch_size=16, tubelet_size=2, in_chans=3, embed_dim=768):
        super().__init__()
        # Single 3D conv that does all the work:
        # kernel_size = (tubelet_size, patch_size, patch_size)
        # stride = same as kernel → non-overlapping patches
        self.proj = nn.Conv3d(
            in_channels=in_chans,
            out_channels=embed_dim,
            kernel_size=(tubelet_size, patch_size, patch_size),
            stride=(tubelet_size, patch_size, patch_size),
        )
    
    def forward(self, x):
        # x: [B, C=3, T, H, W]
        x = self.proj(x)  # [B, D, T/tubelet, H/patch, W/patch]
        B, D, Tp, Hp, Wp = x.shape
        x = x.flatten(2).transpose(1, 2)  # [B, Tp*Hp*Wp, D]
        return x

# Demo: what happens to a video
patch_embed = PatchEmbed3D(patch_size=16, tubelet_size=2, embed_dim=1408)
video = torch.randn(1, 3, 16, 256, 256)  # 1 video, 3 channels, 16 frames, 256x256

tokens = patch_embed(video)
print(f"Input video:  {video.shape}  (B, C, T, H, W)")
print(f"Output tokens: {tokens.shape}  (B, N_patches, D)")
print(f"")
print(f"Breakdown:")
print(f"  Temporal tokens: 16 / 2 = 8")
print(f"  Spatial tokens:  (256/16) × (256/16) = 16 × 16 = 256")
print(f"  Total tokens:    8 × 256 = {8 * 256}")
print(f"  Embedding dim:   1408 (ViT-giant)")

## 2. 3D Rotary Position Embeddings (3D RoPE)

V-JEPA 2 uses **3D RoPE** instead of learned or sinusoidal position embeddings.

### What is RoPE?
Standard RoPE (Rotary Position Embedding) encodes positions by **rotating** query and key vectors.
If two tokens are at positions $m$ and $n$, their attention score naturally depends on $m - n$ (relative position).

### 3D Extension for Video
Each token has 3 position coordinates: **(depth, height, width)**.
The head dimension is split into 3 parts, and each part gets rotated by its corresponding axis:

```
head_dim = dim // num_heads  (e.g., 1408/22 = 64)
d_dim = 2 * ((64 // 3) // 2) = 20  → temporal rotation
h_dim = 20                          → height rotation
w_dim = 20                          → width rotation
residual = 64 - 60 = 4             → unrotated
```

Source: `refs/vjepa2/src/models/utils/modules.py`

In [ ]:
# VISUALIZE RoPE: How rotation encodes position on the unit circle
# RoPE rotates pairs of dimensions by angle θ = position / 10000^(2i/d)

# Local helper for RoPE rotation (defined here so this cell is self-contained)
def _rope_rotate(x, pos):
    """Apply rotary position embedding. x: [B, heads, N, D], pos: positions."""
    B, H, N, D = x.size()
    omega = torch.arange(D // 2, dtype=x.dtype) / (D / 2.0)
    omega = 1.0 / 10000**omega
    freq = torch.einsum('..., f -> ... f', pos, omega)
    emb_sin = freq.sin().unsqueeze(1).repeat(1, H, 1, 2)
    emb_cos = freq.cos().unsqueeze(1).repeat(1, H, 1, 2)
    y = x.unflatten(-1, (-1, 2))
    y1, y2 = y.unbind(dim=-1)
    y = torch.stack((-y2, y1), dim=-1).flatten(-2)
    return (x * emb_cos) + (y * emb_sin)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Rotation angle vs position for different frequency dimensions
ax = axes[0]
D_rope = 20  # number of RoPE dimensions (half of head_dim for one axis)
positions = np.arange(16)  # positions 0-15

for dim_idx in [0, 2, 5, 9]:
    omega = 1.0 / (10000 ** (2 * dim_idx / D_rope))
    angles = positions * omega
    ax.plot(positions, np.degrees(angles) % 360, 'o-', markersize=4, 
            label=f'dim pair {dim_idx}: ω={omega:.4f}')

ax.set_xlabel('Token Position')
ax.set_ylabel('Rotation Angle (degrees)')
ax.set_title('RoPE: Each dimension pair rotates\nat a different frequency')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Plot 2: Unit circle showing rotation for 2 dimension pairs
ax = axes[1]
theta_circle = np.linspace(0, 2*np.pi, 100)
ax.plot(np.cos(theta_circle), np.sin(theta_circle), 'k-', alpha=0.2)

# Dim pair 0 (slow rotation)
omega_slow = 1.0 / (10000 ** (0 / D_rope))
for pos in range(8):
    angle = pos * omega_slow
    x_rot = np.cos(angle)
    y_rot = np.sin(angle)
    ax.plot(x_rot, y_rot, 'ro', markersize=8)
    ax.annotate(f'p={pos}', (x_rot, y_rot), textcoords="offset points", 
                xytext=(5, 5), fontsize=7, color='red')

# Dim pair 5 (fast rotation)
omega_fast = 1.0 / (10000 ** (10 / D_rope))
for pos in range(8):
    angle = pos * omega_fast
    x_rot = np.cos(angle)
    y_rot = np.sin(angle)
    ax.plot(x_rot, y_rot, 'bs', markersize=6)
    ax.annotate(f'p={pos}', (x_rot, y_rot), textcoords="offset points", 
                xytext=(-15, -10), fontsize=7, color='blue')

ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')
ax.set_title('Unit Circle: Slow (red) vs Fast (blue)\ndimension pairs at positions 0-7')
ax.grid(True, alpha=0.3)

# Plot 3: Attention score vs relative position (showing translation invariance)
ax = axes[2]
head_dim_demo = 16
q_base = torch.randn(1, 1, 1, head_dim_demo)

rel_positions = np.arange(-8, 9)
attn_scores = []

for rel_pos in rel_positions:
    q_pos = 8
    k_pos = q_pos + rel_pos
    if k_pos < 0 or k_pos > 16:
        attn_scores.append(0)
        continue
    
    # Apply RoPE using local helper
    q_rotated = _rope_rotate(q_base, torch.tensor([[float(q_pos)]]))
    k_rotated = _rope_rotate(q_base.clone(), torch.tensor([[float(k_pos)]]))
    
    score = (q_rotated @ k_rotated.transpose(-2, -1)).item() / (head_dim_demo ** 0.5)
    attn_scores.append(score)

ax.bar(rel_positions, attn_scores, color=['#e74c3c' if s < 0 else '#3498db' for s in attn_scores])
ax.set_xlabel('Relative Position (key - query)')
ax.set_ylabel('Attention Score (before softmax)')
ax.set_title('RoPE: Attention depends on\nRELATIVE position (m-n)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("RoPE key properties:")
print("1. Rotation angle θ = position × frequency (like Fourier basis)")
print("2. Inner product q·k depends only on relative position (m-n)")
print("3. Low-frequency dims capture coarse position, high-frequency dims capture fine position")
print("4. 3D RoPE: split head_dim into 3 axes, apply RoPE independently per axis")

In [ ]:
def rotate_queries_or_keys(x, pos):
    """
    Apply rotary position embeddings to queries or keys.
    
    Source: refs/vjepa2/src/models/utils/modules.py, line 26-55
    
    x:   [B, num_heads, N, D]  — query or key vectors
    pos: [N] or [B, num_heads, N]  — position indices
    
    Math:
      For each pair (x_{2i}, x_{2i+1}) at position p:
        θ_i = p / 10000^(2i/D)
        x_{2i}'   = x_{2i} * cos(θ_i) - x_{2i+1} * sin(θ_i)
        x_{2i+1}' = x_{2i+1} * cos(θ_i) + x_{2i} * sin(θ_i)
    """
    B, num_heads, N, D = x.size()
    assert D % 2 == 0
    
    # Compute frequency for each dimension pair
    omega = torch.arange(D // 2, dtype=x.dtype, device=x.device)
    omega /= D / 2.0
    omega = 1.0 / 10000**omega  # [D/2] — frequencies
    
    # Outer product: position × frequency → angle
    freq = torch.einsum('..., f -> ... f', pos, omega)  # [..., N, D/2]
    
    emb_sin = freq.sin().squeeze(-1).repeat(1, 1, 1, 2)  # [..., N, D]
    emb_cos = freq.cos().squeeze(-1).repeat(1, 1, 1, 2)  # [..., N, D]
    
    # Rotate: swap pairs and negate
    y = x.unflatten(-1, (-1, 2))       # [..., N, D/2, 2]
    y1, y2 = y.unbind(dim=-1)           # each [..., N, D/2]
    y = torch.stack((-y2, y1), dim=-1)  # [-y2, y1] pairs
    y = y.flatten(-2)                   # [..., N, D]
    
    return (x * emb_cos) + (y * emb_sin)

# Demo: how rotation encodes position
B, H_heads, N, D = 1, 1, 4, 8
q = torch.randn(B, H_heads, N, D)
positions = torch.tensor([0.0, 1.0, 2.0, 3.0])  # 4 tokens at positions 0,1,2,3

q_rotated = rotate_queries_or_keys(q, positions)
print(f"Original q[0,0,0]: {q[0,0,0,:4].tolist()}")
print(f"Rotated  q[0,0,0]: {q_rotated[0,0,0,:4].tolist()}")
print(f"")
print("The rotation depends on position — same vector at different positions gets different rotations.")
print("This means attention(q_m, k_n) naturally depends on relative position (m-n).")

In [ ]:
def demonstrate_3d_rope_splitting():
    """
    Show how V-JEPA 2 splits the head dimension for 3D RoPE.
    Source: RoPEAttention.__init__ in modules.py, lines 290-293
    """
    # ViT-giant: embed_dim=1408, num_heads=22
    for name, embed_dim, num_heads in [
        ("ViT-Large", 1024, 16),
        ("ViT-Huge", 1280, 16),
        ("ViT-Giant", 1408, 22),
    ]:
        head_dim = embed_dim // num_heads
        d_dim = int(2 * ((head_dim // 3) // 2))  # temporal (depth)
        h_dim = int(2 * ((head_dim // 3) // 2))  # height
        w_dim = int(2 * ((head_dim // 3) // 2))  # width
        residual = head_dim - d_dim - h_dim - w_dim
        
        print(f"{name}: embed={embed_dim}, heads={num_heads}, head_dim={head_dim}")
        print(f"  depth_dim={d_dim}, height_dim={h_dim}, width_dim={w_dim}, residual={residual}")
        print(f"  → {d_dim+h_dim+w_dim}/{head_dim} dims rotated, {residual} unrotated")
        print()

demonstrate_3d_rope_splitting()

### 3.5 Self-Attention: The Matrix Math in Detail

The attention mechanism is a **weighted information aggregation**. Here's every matrix operation:

**Step 1 — QKV Projection (one linear layer, split into 3):**
$$[\mathbf{Q}; \mathbf{K}; \mathbf{V}] = \mathbf{X} \mathbf{W}_{QKV} \in \mathbb{R}^{N \times 3D}$$
$$\mathbf{Q}, \mathbf{K}, \mathbf{V} \in \mathbb{R}^{N \times D}, \quad \text{then reshape to } [N_{\text{heads}}, N, d_{\text{head}}]$$

**Step 2 — Apply 3D RoPE (rotation matrices per position):**
$$\mathbf{Q}' = \mathbf{R}_{3D}(\text{pos}) \odot \mathbf{Q}, \quad \mathbf{K}' = \mathbf{R}_{3D}(\text{pos}) \odot \mathbf{K}$$
where $\mathbf{R}_{3D}$ applies independent rotations for depth, height, width axes.

**Step 3 — Attention scores (N×N matrix per head):**
$$\mathbf{A} = \text{softmax}\!\left(\frac{\mathbf{Q}' {\mathbf{K}'}^T}{\sqrt{d_{\text{head}}}}\right) \in \mathbb{R}^{N \times N}$$

Each entry $A_{ij}$ = "how much should token $i$ attend to token $j$?"

**Step 4 — Weighted value aggregation:**
$$\mathbf{O} = \mathbf{A} \mathbf{V} \in \mathbb{R}^{N \times d_{\text{head}}}$$

Each output token is a weighted average of ALL value vectors.

**Step 5 — Concatenate heads and project:**
$$\text{Output} = \text{Concat}(\mathbf{O}_1, ..., \mathbf{O}_H) \mathbf{W}_{\text{proj}} \in \mathbb{R}^{N \times D}$$

**FLOPs for one attention layer (V-JEPA 2, N=205 visible tokens):**
- QKV projection: $3 \times N \times D^2 = 3 \times 205 \times 1408^2 \approx 1.2$B FLOPs
- Attention scores: $N^2 \times D = 205^2 \times 1408 \approx 59$M FLOPs
- Value aggregation: $N^2 \times D = 59$M FLOPs
- Output projection: $N \times D^2 = 205 \times 1408^2 \approx 406$M FLOPs

Total per block: ~1.7B FLOPs. For 40 blocks: ~68B FLOPs.
Compare to processing ALL 2048 tokens: ~680B FLOPs → **10x savings from masking!**

In [ ]:
def visualize_3d_positions(T=8, H=16, W=16):
    """
    Visualize how V-JEPA 2 separates a flat token index into (depth, height, width).
    Source: RoPEAttention.separate_positions() in modules.py, lines 316-329
    """
    # Flat indices for all tokens
    ids = torch.arange(T * H * W)
    
    # Separate into 3D coordinates (this is the actual code logic)
    tokens_per_frame = H * W         # 256
    tokens_per_row = W               # 16
    
    frame_ids = ids // tokens_per_frame                       # depth (temporal)
    height_ids = (ids - tokens_per_frame * frame_ids) // tokens_per_row  # height
    width_ids = (ids - tokens_per_frame * frame_ids) - tokens_per_row * height_ids  # width
    
    # Show some examples
    print(f"Total tokens: {T}×{H}×{W} = {T*H*W}")
    print(f"")
    print(f"{'Flat Index':<12} {'Frame (d)':<12} {'Row (h)':<12} {'Col (w)':<12}")
    print("-" * 48)
    for idx in [0, 1, 15, 16, 255, 256, 257, 2047]:
        if idx < len(ids):
            print(f"{idx:<12} {frame_ids[idx].item():<12} {height_ids[idx].item():<12} {width_ids[idx].item():<12}")
    
    return frame_ids, height_ids, width_ids

d, h, w = visualize_3d_positions()
print(f"\nThese 3 position vectors are used for the 3 RoPE rotation axes.")
print(f"Each axis gets its own rotation → q and k encode 3D spatial-temporal structure.")

In [ ]:
# ATTENTION MATRIX VISUALIZATION
# Show what the attention matrix looks like and how it aggregates information

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Create a small attention example
N_tokens = 12
D_attn = 16
torch.manual_seed(42)

Q = torch.randn(1, 1, N_tokens, D_attn)
K = torch.randn(1, 1, N_tokens, D_attn)
V = torch.randn(1, 1, N_tokens, D_attn)

# Raw attention scores
scores = (Q @ K.transpose(-2, -1)) / (D_attn ** 0.5)
attn_weights = scores.softmax(dim=-1)

# Plot 1: Raw scores (before softmax)
ax = axes[0]
im = ax.imshow(scores[0, 0].detach().numpy(), cmap='RdBu', vmin=-2, vmax=2)
plt.colorbar(im, ax=ax)
ax.set_xlabel('Key position')
ax.set_ylabel('Query position')
ax.set_title('Raw Attention Scores\n(Q·K^T / √d, before softmax)')

# Plot 2: Attention weights (after softmax)
ax = axes[1]
im = ax.imshow(attn_weights[0, 0].detach().numpy(), cmap='Oranges', vmin=0, vmax=0.3)
plt.colorbar(im, ax=ax)
ax.set_xlabel('Key position')
ax.set_ylabel('Query position')
ax.set_title('Attention Weights\n(after softmax, rows sum to 1)')

# Verify rows sum to 1
row_sums = attn_weights[0, 0].sum(dim=-1)

# Plot 3: Output = weighted sum of values
output = attn_weights @ V
ax = axes[2]
im = ax.imshow(output[0, 0].detach().numpy(), cmap='viridis', aspect='auto')
plt.colorbar(im, ax=ax)
ax.set_xlabel('Embedding dimension')
ax.set_ylabel('Token position')
ax.set_title('Attention Output\n(each row = weighted avg of V)')

plt.tight_layout()
plt.show()

print(f"Attention matrix shape: {attn_weights.shape} → [{N_tokens} queries × {N_tokens} keys]")
print(f"Each row sums to 1.0: min={row_sums.min():.4f}, max={row_sums.max():.4f}")
print(f"Output = Attention × V: [{N_tokens} tokens × {D_attn} dim]")
print(f"\nIn V-JEPA 2 (visible tokens only): [{205} queries × {205} keys] per head, {22} heads")

In [ ]:
# SwiGLU vs GELU: Detailed activation comparison
# Why does V-JEPA 2 use SwiGLU? It's a GATED activation — selectively passes information

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

x = torch.linspace(-4, 4, 200)

# Plot 1: Activation functions
ax = axes[0]
ax.plot(x.numpy(), F.relu(x).numpy(), '--', linewidth=1.5, label='ReLU', alpha=0.7)
ax.plot(x.numpy(), F.gelu(x).numpy(), '-', linewidth=2, label='GELU')
ax.plot(x.numpy(), F.silu(x).numpy(), '-', linewidth=2, label='SiLU (Swish)')
ax.set_xlabel('Input x')
ax.set_ylabel('Output')
ax.set_title('Activation Functions')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Standard MLP vs SwiGLU forward pass
ax = axes[1]
dim = 8
hidden = 16

# Standard: GELU(W1·x)
W1 = torch.randn(dim, hidden) * 0.3
input_vec = torch.randn(1, dim)
std_hidden = F.gelu(input_vec @ W1)

# SwiGLU: SiLU(W1·x) * W2·x
W2 = torch.randn(dim, hidden) * 0.3
gate = F.silu(input_vec @ W1)
value = input_vec @ W2
swiglu_hidden = gate * value

ax.bar(np.arange(hidden) - 0.15, std_hidden[0].detach().numpy(), 0.3, 
       label='GELU(W1·x)', color='#3498db', alpha=0.7)
ax.bar(np.arange(hidden) + 0.15, swiglu_hidden[0].detach().numpy(), 0.3, 
       label='SiLU(W1·x)·W2·x', color='#e74c3c', alpha=0.7)
ax.set_xlabel('Hidden dimension')
ax.set_ylabel('Activation value')
ax.set_title('Standard MLP vs SwiGLU\n(SwiGLU gates can suppress dimensions)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Plot 3: The gating mechanism
ax = axes[2]
gate_vals = gate[0].detach().numpy()
value_vals = value[0].detach().numpy()
product = (gate * value)[0].detach().numpy()

x_pos = np.arange(hidden)
ax.bar(x_pos - 0.2, np.abs(gate_vals), 0.2, label='|gate|: SiLU(W1·x)', 
       color='#2ecc71', alpha=0.7)
ax.bar(x_pos, np.abs(value_vals), 0.2, label='|value|: W2·x', 
       color='#3498db', alpha=0.7)
ax.bar(x_pos + 0.2, np.abs(product), 0.2, label='|output|: gate × value', 
       color='#e74c3c', alpha=0.7)
ax.set_xlabel('Hidden dimension')
ax.set_ylabel('|Activation|')
ax.set_title('SwiGLU Gating: gate selectively\npasses or blocks each dimension')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("SwiGLU: out = SiLU(W1·x) × W2·x")
print("  - W1 path (gate): decides WHICH dimensions are active (via SiLU activation)")
print("  - W2 path (value): provides the actual information content")
print("  - Multiplication: gate selectively passes/blocks information")
print("  - This gives the network more control over information flow")
print("  - Uses 3 weight matrices (W1, W2, W3) vs 2 for standard MLP")
print(f"  - But with wide_silu adjustment: hidden_dim = align_to_8(2/3 × mlp_ratio × dim)")
print(f"  - So total params are similar to standard MLP")

## 3. Transformer Block: The Building Block

V-JEPA 2's transformer block follows pre-norm architecture:

```
x = x + DropPath(Attention(LayerNorm(x)))  ← with 3D RoPE
x = x + DropPath(MLP(LayerNorm(x)))        ← SwiGLU FFN for ViT-g
```

The ViT-giant uses **SwiGLU** instead of standard GELU MLP:

$$\text{SwiGLU}(x) = (\text{SiLU}(W_1 x)) \odot (W_2 x)$$
$$\text{out} = W_3 \cdot \text{SwiGLU}(x)$$

Source: `refs/vjepa2/src/models/utils/modules.py`, SwiGLUFFN class

In [ ]:
class SwiGLUFFN(nn.Module):
    """SwiGLU Feed-Forward Network.
    
    Used in V-JEPA 2 ViT-giant instead of standard GELU MLP.
    Source: refs/vjepa2/src/models/utils/modules.py, lines 91-111
    
    SwiGLU(x) = SiLU(W1·x) * W2·x   ← gated activation
    out = W3 · SwiGLU(x)
    
    With wide_silu=True (default), hidden dim is adjusted:
    hidden = align_to_8(2/3 * mlp_ratio * dim)
    """
    def __init__(self, dim, hidden_dim, wide_silu=True):
        super().__init__()
        if wide_silu:
            swiglu_hidden = int(2 * hidden_dim / 3)
            swiglu_hidden = (swiglu_hidden + 7) // 8 * 8  # align to 8
        else:
            swiglu_hidden = hidden_dim
        
        self.fc1 = nn.Linear(dim, swiglu_hidden)   # W1: gate
        self.fc2 = nn.Linear(dim, swiglu_hidden)   # W2: value
        self.fc3 = nn.Linear(swiglu_hidden, dim)   # W3: project back
    
    def forward(self, x):
        x1 = self.fc1(x)          # gate path
        x2 = self.fc2(x)          # value path
        hidden = F.silu(x1) * x2  # SwiGLU: gated activation
        return self.fc3(hidden)

# Compare parameter counts: Standard MLP vs SwiGLU
dim = 1408  # ViT-giant embed dim
mlp_ratio = 48/11  # ViT-giant mlp_ratio
hidden = int(dim * mlp_ratio)

# Standard MLP: 2 linear layers
std_params = dim * hidden + hidden * dim

# SwiGLU: 3 linear layers but smaller hidden
swiglu_hidden = int(2 * hidden / 3)
swiglu_hidden = (swiglu_hidden + 7) // 8 * 8
swi_params = dim * swiglu_hidden * 2 + swiglu_hidden * dim

print(f"ViT-giant (dim={dim}, mlp_ratio={mlp_ratio:.2f}):")
print(f"  Standard MLP hidden dim: {hidden}, params: {std_params:,}")
print(f"  SwiGLU FFN hidden dim:   {swiglu_hidden}, params: {swi_params:,}")
print(f"  SwiGLU/Standard ratio:   {swi_params/std_params:.2f}x")

## 4. Weight Initialization: Rescaling Trick

V-JEPA 2 uses an important initialization trick from the DeepNet paper:

```python
# refs/vjepa2/src/models/vision_transformer.py, lines 147-153
def _rescale_blocks(self):
    def rescale(param, layer_id):
        param.div_(math.sqrt(2.0 * layer_id))
    for layer_id, layer in enumerate(self.blocks):
        rescale(layer.attn.proj.weight.data, layer_id + 1)
        rescale(layer.mlp.fc2.weight.data, layer_id + 1)
```

This divides the output projection weights of each block by $\sqrt{2L}$ where $L$ is the layer index.
For a 40-layer ViT-giant, the last layer's weights are divided by $\sqrt{80} \approx 8.9$.

**Why?** Deep residual networks accumulate variance through skip connections.
Without rescaling, the output variance grows as $O(L)$, which can cause training instability.
This rescaling ensures each layer contributes roughly equally.

In [ ]:
# Visualize the rescaling factor across layers
depths = [24, 32, 40]  # ViT-L, ViT-H, ViT-g
names = ['ViT-Large (24)', 'ViT-Huge (32)', 'ViT-Giant (40)']

fig, ax = plt.subplots(figsize=(10, 5))
for depth, name in zip(depths, names):
    layers = list(range(1, depth + 1))
    scale_factors = [1.0 / math.sqrt(2.0 * l) for l in layers]
    ax.plot(layers, scale_factors, '-o', markersize=3, label=name)

ax.set_xlabel('Layer Index')
ax.set_ylabel('Weight Scale Factor (1 / √(2L))')
ax.set_title('V-JEPA 2 Weight Rescaling: Deeper layers have smaller initial weights')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"ViT-Giant layer 1:  weights scaled by {1/math.sqrt(2):.3f}")
print(f"ViT-Giant layer 40: weights scaled by {1/math.sqrt(80):.3f}")
print(f"→ Last layer's initial output is ~{math.sqrt(80)/math.sqrt(2):.1f}x smaller than first layer's")

## 5. Putting It Together: Mini V-JEPA 2 (CPU-Runnable)

Let's build a small but faithful V-JEPA 2 that we can actually run on CPU.
We'll use ViT-tiny dimensions but the exact same code structure.

In [ ]:
class MiniRoPEAttention(nn.Module):
    """Simplified 3D RoPE Attention matching V-JEPA 2 structure."""
    def __init__(self, dim, num_heads, grid_size=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        
        # 3D RoPE dimensions
        self.d_dim = int(2 * ((self.head_dim // 3) // 2))
        self.h_dim = int(2 * ((self.head_dim // 3) // 2))
        self.w_dim = int(2 * ((self.head_dim // 3) // 2))
        self.grid_size = grid_size
    
    def separate_positions(self, ids, H, W):
        tpf = H * W
        tpr = W
        d = ids // tpf
        h = (ids - tpf * d) // tpr
        w = (ids - tpf * d) - tpr * h
        return d.float(), h.float(), w.float()
    
    def forward(self, x, mask=None, T=None, H_patches=None, W_patches=None):
        B, N, C = x.shape
        qkv = self.qkv(x).unflatten(-1, (3, self.num_heads, -1)).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        # Position IDs
        if mask is not None:
            pos_ids = mask.unsqueeze(1).repeat(1, self.num_heads, 1)
        else:
            pos_ids = torch.arange(N, device=x.device)
        d_mask, h_mask, w_mask = self.separate_positions(pos_ids, H_patches or self.grid_size, W_patches or self.grid_size)
        
        # Apply 3D RoPE: rotate each axis separately
        s = 0
        qd = rotate_queries_or_keys(q[..., s:s+self.d_dim], pos=d_mask)
        kd = rotate_queries_or_keys(k[..., s:s+self.d_dim], pos=d_mask)
        s += self.d_dim
        qh = rotate_queries_or_keys(q[..., s:s+self.h_dim], pos=h_mask)
        kh = rotate_queries_or_keys(k[..., s:s+self.h_dim], pos=h_mask)
        s += self.h_dim
        qw = rotate_queries_or_keys(q[..., s:s+self.w_dim], pos=w_mask)
        kw = rotate_queries_or_keys(k[..., s:s+self.w_dim], pos=w_mask)
        s += self.w_dim
        
        if s < self.head_dim:
            q = torch.cat([qd, qh, qw, q[..., s:]], dim=-1)
            k = torch.cat([kd, kh, kw, k[..., s:]], dim=-1)
        else:
            q = torch.cat([qd, qh, qw], dim=-1)
            k = torch.cat([kd, kh, kw], dim=-1)
        
        # Standard attention
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(x)


class MiniBlock(nn.Module):
    """Transformer block matching V-JEPA 2 structure."""
    def __init__(self, dim, num_heads, mlp_ratio=4.0, grid_size=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MiniRoPEAttention(dim, num_heads, grid_size)
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = SwiGLUFFN(dim, hidden)
    
    def forward(self, x, mask=None, T=None, H_patches=None, W_patches=None):
        x = x + self.attn(self.norm1(x), mask=mask, T=T, H_patches=H_patches, W_patches=W_patches)
        x = x + self.mlp(self.norm2(x))
        return x


print("Mini V-JEPA 2 building blocks created!")
print(f"These use the exact same 3D RoPE and SwiGLU as the real V-JEPA 2.")

## 6. The Predictor: How It Works

The predictor's job:
1. Take encoder output (visible patches only)
2. Add **learnable mask tokens** at target positions
3. Sort everything by position
4. Process through transformer blocks
5. Extract only the mask token outputs
6. Project back to encoder dimension

Source: `refs/vjepa2/src/models/predictor.py`

```
Context tokens: [c₀, c₅, c₁₀]  (visible patches at positions 0, 5, 10)
Mask tokens:    [m₁, m₂, m₃, m₄, m₆, m₇, m₈, m₉, m₁₁, ...]  (at masked positions)

Concatenate & sort by position:
[c₀, m₁, m₂, m₃, m₄, c₅, m₆, m₇, m₈, m₉, c₁₀, m₁₁, ...]

Process through transformer → all tokens attend to each other

Extract mask tokens → project to encoder dim → these are the predictions
```

In [ ]:
# Demonstrate the predictor's token flow
N_total = 16  # total patches
mask_ratio = 0.75
N_visible = int(N_total * (1 - mask_ratio))  # 4
N_masked = N_total - N_visible  # 12

# Simulate masks
perm = torch.randperm(N_total)
visible_idx = perm[:N_visible].sort()[0]
masked_idx = perm[N_visible:].sort()[0]

print(f"Total patches: {N_total}")
print(f"Visible (encoder sees): {visible_idx.tolist()} ({N_visible} patches)")
print(f"Masked (predictor predicts): {masked_idx.tolist()} ({N_masked} patches)")
print()

# Step 1: Encoder outputs for visible patches
encoder_output = torch.randn(1, N_visible, 64)  # [B, 4, D]
print(f"Step 1 - Encoder output shape: {encoder_output.shape}")

# Step 2: Project to predictor dimension
predictor_dim = 32
projected = torch.randn(1, N_visible, predictor_dim)  # simulated
print(f"Step 2 - Projected to predictor dim: {projected.shape}")

# Step 3: Create mask tokens at target positions
mask_tokens = torch.zeros(1, N_masked, predictor_dim)  # learnable, initialized to zero
print(f"Step 3 - Mask tokens: {mask_tokens.shape}")

# Step 4: Concatenate
all_tokens = torch.cat([projected, mask_tokens], dim=1)  # [1, 16, 32]
all_indices = torch.cat([visible_idx, masked_idx])  # [16]
print(f"Step 4 - Concatenated: {all_tokens.shape}, indices: {all_indices.tolist()}")

# Step 5: Sort by position
sort_order = torch.argsort(all_indices)
sorted_tokens = all_tokens[:, sort_order]
sorted_indices = all_indices[sort_order]
print(f"Step 5 - Sorted by position: {sorted_indices.tolist()}")

# Step 6: After processing, extract mask tokens using reverse sort
reverse_sort = torch.argsort(sort_order)
unsorted = sorted_tokens[:, reverse_sort]  # back to original order
mask_predictions = unsorted[:, N_visible:]  # extract mask token outputs
print(f"Step 6 - Extracted predictions: {mask_predictions.shape} → project back to encoder dim")

## 7. Model Sizes: V-JEPA 2 Family

From `refs/vjepa2/src/models/vision_transformer.py`:

In [ ]:
def compute_vit_params(embed_dim, depth, num_heads, mlp_ratio, use_swiglu=True):
    """Estimate parameter count for a ViT encoder."""
    # Attention: QKV + output projection
    attn_params = 4 * embed_dim * embed_dim  # qkv (3x) + proj (1x)
    
    # FFN
    hidden = int(embed_dim * mlp_ratio)
    if use_swiglu:
        swiglu_hidden = int(2 * hidden / 3)
        swiglu_hidden = (swiglu_hidden + 7) // 8 * 8
        ffn_params = 2 * embed_dim * swiglu_hidden + swiglu_hidden * embed_dim  # fc1, fc2, fc3
    else:
        ffn_params = 2 * embed_dim * hidden  # fc1 + fc2
    
    # LayerNorm: 2 per block
    ln_params = 4 * embed_dim
    
    # Per block
    block_params = attn_params + ffn_params + ln_params
    
    # Patch embedding (Conv3d): 3 * 2 * 16 * 16 * embed_dim
    patch_params = 3 * 2 * 16 * 16 * embed_dim
    
    total = depth * block_params + patch_params
    return total

models = [
    ("ViT-Large",  1024, 24, 16, 4.0, False),
    ("ViT-Huge",   1280, 32, 16, 4.0, False),
    ("ViT-Giant",  1408, 40, 22, 48/11, True),  # V-JEPA 2's main model
]

print(f"{'Model':<12} {'embed_dim':<10} {'depth':<6} {'heads':<6} {'~Params':<12} {'Notes'}")
print("-" * 70)
for name, d, depth, heads, ratio, swiglu in models:
    params = compute_vit_params(d, depth, heads, ratio, swiglu)
    note = "★ V-JEPA 2 main" if name == "ViT-Giant" else ""
    print(f"{name:<12} {d:<10} {depth:<6} {heads:<6} {params/1e6:>8.0f}M    {note}")

# Predictor
pred_params = compute_vit_params(384, 12, 12, 4.0, False)
print(f"")
print(f"{'Predictor':<12} {'384':<10} {'12':<6} {'12':<6} {pred_params/1e6:>8.0f}M    (pretraining predictor)")
ac_pred_params = compute_vit_params(1024, 24, 16, 4.0, False)
print(f"{'AC-Pred':<12} {'1024':<10} {'24':<6} {'16':<6} {ac_pred_params/1e6:>8.0f}M    (action-conditioned, robotics)")

## 8. Summary: V-JEPA 2 Architecture Key Points

### Encoder (ViT-giant)
- **40 transformer blocks**, embed_dim=1408, 22 attention heads
- **3D RoPE** position encoding (depth/height/width axes)
- **SwiGLU FFN** instead of standard GELU MLP
- **DeepNet rescaling** initialization
- Only processes **~10% visible patches** (extremely efficient!)

### Predictor (ViT-small-like)
- **12 blocks**, embed_dim=384, 12 heads (much smaller than encoder)
- Takes encoder output + **learnable mask tokens**
- Sorts tokens by position → processes → extracts mask token outputs
- Projects back to encoder dimension for loss computation

### Target Encoder
- **Identical to encoder** but weights updated via EMA (τ=0.99925)
- Processes **ALL patches** (no masking) → provides prediction targets
- **Stop-gradient** → prevents representation collapse
- Outputs are **LayerNorm'd** before serving as targets

### Training
- 1M+ hours video, 128 GPUs, 800 epochs
- AdamW optimizer, lr=0.000525, wd=0.04
- bfloat16 mixed precision
- Progressive: 16 frames @ 256px → cooldown at 64 frames @ 384px

---

**Next notebook:** V-JEPA 2-AC — how actions condition the predictor for robotics (world model + planning).